In [0]:
%pip install -U python-dotenv openai langchain-openai langchain-chroma langchain-huggingface langchain-core sentence-transformers gradio
dbutils.library.restartPython()

## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

### Prerequisites

This notebook requires:
1. **Vector Store from day2**: The `vector_db` directory created in day2 with 413 embedded chunks
2. **Same embedding model**: HuggingFace `all-MiniLM-L6-v2` (384 dimensions)
3. **OpenAI API Key**: Set `OPENAI_API_KEY` in your `.env` file for the chat model
4. **Installed packages**: sentence-transformers, gradio, langchain libraries (installed in cell 1)

**Note**: Make sure day2 notebook has been run first to create the vector store!

In [0]:
from dotenv import load_dotenv
from openai import OpenAI

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [0]:
import os

DB_NAME = "vector_db"
load_dotenv(override=True)

# Check if OpenAI API key is available
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key found (begins with {openai_api_key[:8]}...)")
    MODEL = "gpt-4o-mini"  # Fast and cost-effective OpenAI model
    openai = OpenAI()
else:
    print("OpenAI API Key not set - using Databricks AI Gateway")
    MODEL = "databricks-gpt-oss-120b"  # Databricks free tier model
    # Set up Databricks AI Gateway client with workspace-specific URL
    try:
        databricks_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    except:
        databricks_token = os.environ.get("DATABRICKS_TOKEN", "dummy-token")
    
    openai = OpenAI(
        api_key=databricks_token,
        base_url="https://7474647277163805.ai-gateway.cloud.databricks.com/mlflow/v1"
    )

print(f"Using model: {MODEL}")

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [0]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [0]:
# Set up the retriever from the vector store
retriever = vectorstore.as_retriever()

print(f"Retriever ready using model: {MODEL}")

### These LangChain objects implement the method `invoke()`

In [0]:
retriever.invoke("Who is Avery?")

In [0]:
# Test the LLM directly using openai.chat.completions.create()
messages = [{"role": "user", "content": "Who is Avery?"}]
response = openai.chat.completions.create(model=MODEL, messages=messages)
content = response.choices[0].message.content

# Handle Databricks structured response format (same as day1)
if isinstance(content, list):
    # Extract text from structured response
    for item in content:
        if isinstance(item, dict) and item.get('type') == 'text':
            print(item.get('text', ''))
            break
else:
    # Handle standard OpenAI text response
    print(content)

## Time to put this together!

In [0]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [0]:
def answer_question(question: str, history):
    # Retrieve relevant documents
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    
    # Build the system prompt with context
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    
    # Create messages array for the API call
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    
    # Call OpenAI API directly (works with both OpenAI and Databricks)
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    content = response.choices[0].message.content
    
    # Handle Databricks structured response format (from day1/day2)
    if isinstance(content, list):
        # Extract text from structured response
        for item in content:
            if isinstance(item, dict) and item.get('type') == 'text':
                return item.get('text', '')
        return str(content)  # Fallback if format unexpected
    
    # Handle standard OpenAI text response
    return content

In [0]:
answer_question("Who is Averi Lancaster?", [])

## What could possibly come next? 😂